<!--
  Licensed to the Apache Software Foundation (ASF) under one or more
  contributor license agreements.  See the NOTICE file distributed with
  this work for additional information regarding copyright ownership.
  The ASF licenses this file to You under the Apache License, Version 2.0
  (the "License"); you may not use this file except in compliance with
  the License.  You may obtain a copy of the License at

       http://www.apache.org/licenses/LICENSE-2.0

  Unless required by applicable law or agreed to in writing, software
  distributed under the License is distributed on an "AS IS" BASIS,
  WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
  See the License for the specific language governing permissions and
  limitations under the License.
-->

# Hudi VECTOR + BLOB + Vector Search — SQL Demo

This is the **canonical demo** for the two Hudi 1.2.0 logical types working
together end-to-end:

- **`VECTOR(N)`** — fixed-length float embeddings as a first-class column
  type, queryable through the **`hudi_vector_search`** SQL TVF.
- **`BLOB`** — bytes (or references to external bytes) as a first-class
  column type, materialized through the **`read_blob()`** SQL function.

Every Hudi-touching operation is a SQL string so the actual DDL/DML surface
is visible:

- `CREATE TABLE … (embedding VECTOR(N), image_bytes BLOB, …) USING hudi`
- `INSERT INTO … SELECT …, named_struct('type', 'INLINE', …)`
- `SELECT read_blob(image_bytes), … FROM hudi_vector_search('<path>', 'embedding', ARRAY(…), k, 'cosine')`

The last query is the punchline — it composes both features in a single
SELECT. `hudi_vector_search` finds the top-K nearest neighbors;
`read_blob()` materializes the BLOB column to raw bytes for visualization.
Format-agnostic: works against both Parquet and Lance base files, and
against both INLINE and OUT_OF_LINE BLOBs (this notebook uses INLINE; see
[`01_blob_reader.ipynb`](01_blob_reader.ipynb) for the OUT_OF_LINE /
DESCRIPTOR deep-dive).

Image loading (torchvision) and embedding generation (timm) stay in
Python — those can't be SQL. The bridge to Spark is a temp view.

## 1. Toggles

**What this cell does:** sets the four knobs that control this run.

| Variable | Effect |
|---|---|
| `BASE_FILE_FORMAT` | `"parquet"` or `"lance"` — flips Hudi's base file format. No other code changes; the same DDL/DML works against both. |
| `N_SAMPLES`        | How many Oxford-IIIT Pet images to load. Default 250 keeps live demos under a minute; bump to 1000+ for a production-feel run (RAM/heap permitting). |
| `TOP_K`            | How many nearest neighbors to retrieve from `hudi_vector_search`. |
| `EMBEDDING_MODEL`  | The `timm` model used to generate image embeddings. The default (`mobilenetv3_small_100`) is small and CPU-friendly. |

Edit and re-run **just this cell + Run All Below** to reconfigure without
restarting Spark.

In [1]:
# ===== EDIT THESE =====
BASE_FILE_FORMAT = "parquet"   # "parquet" or "lance"
N_SAMPLES        = 250
TOP_K            = 5
EMBEDDING_MODEL  = "mobilenetv3_small_100"

assert BASE_FILE_FORMAT in {"parquet", "lance"}

## 2. Pre-flight cleanup

**What this cell does:** wipes the `/tmp/` paths and `spark-warehouse/`
directory that previous runs of this notebook (or the sibling `.py` scripts)
left behind, so each run starts from a clean slate.

**Why:** `DROP TABLE IF EXISTS` only removes the catalog entry — the data
dir and `.hoodie/` timeline at the table's `LOCATION` persist across runs,
so a re-run would see old commits alongside new ones. The shared
`/tmp/pets_blob_container.bin` (used by `01_blob_reader.ipynb`) is also
overwritten on every run, leaving stale tables pointing past EOF
(→ file boundary errors during read). Cleanup at the top makes
every Run All idempotent.

In [ ]:
# Wipe leftover Hudi/Spark state from previous runs.
#
# Why: `DROP TABLE IF EXISTS` only removes the catalog entry — the data dir
# and `.hoodie/` timeline at LOCATION persist across runs. A re-run would
# see old commits alongside new ones, and for the blob_reader notebook the
# shared `/tmp/pets_blob_container.bin` is overwritten by every blob_mode
# run, which leaves stale tables pointing past EOF (→ file boundary errors
# during read). Cleaning up at the top makes every cell run idempotent.
import shutil
from pathlib import Path

for pattern in [
    "/tmp/hudi_*_pets",
    "/tmp/pets_blob_container.bin",
    "/tmp/staging_pets_*.parquet",
]:
    for p in Path("/").glob(pattern.lstrip("/")):
        if p.is_dir():
            shutil.rmtree(p, ignore_errors=True)
        elif p.is_file():
            p.unlink(missing_ok=True)

shutil.rmtree("spark-warehouse", ignore_errors=True)
print("✓ Wiped /tmp/hudi_*_pets, /tmp/pets_blob_container.bin, staging Parquet, spark-warehouse")

## 3. Pre-JVM env (driver heap)

**What this cell does:** sets `PYSPARK_SUBMIT_ARGS` so the JVM driver
launches with `--driver-memory 4g`.

**Why this is in its own cell, before imports:** PySpark in local mode bakes
the driver heap into the JVM at launch time. By the time `pyspark` is
imported (next cell), it's too late — `SparkSession.config(...)` cannot grow
an already-launched JVM. Bump `DRIVER_MEMORY` to `"8g"` or `"12g"` for
N ≥ 2000.

In [3]:
# Pre-JVM env: must run BEFORE any `pyspark` import so the driver heap is
# set at JVM launch time. PySpark in local mode can't grow the heap later
# via SparkSession.config().
import os

DRIVER_MEMORY = "4g"   # bump to "8g" or "12g" for N >= 2000

os.environ.setdefault(
    "PYSPARK_SUBMIT_ARGS",
    f"--driver-memory {DRIVER_MEMORY} --conf spark.driver.maxResultSize=2g pyspark-shell",
)

'--driver-memory 4g --conf spark.driver.maxResultSize=2g pyspark-shell'

## 4. Imports

**What this cell does:** imports the libraries this notebook needs, grouped
by purpose:

- **Data** — `numpy`, `pyarrow` (+ `parquet`), `PIL`, `torchvision.OxfordIIITPet`
- **Embeddings** — `torch`, `timm`, `sklearn.preprocessing.normalize`
- **Spark** — `pyspark.sql.SparkSession`
- **Visualization** — `matplotlib` (Agg backend, no display server) + `IPython.display`

Must run *after* cell 3 (`PYSPARK_SUBMIT_ARGS`) and *before* anything that
touches `pyspark`.

In [4]:
import io
from pathlib import Path

import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq
import torch
import timm
from sklearn.preprocessing import normalize
from PIL import Image

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from torchvision.datasets import OxfordIIITPet

from pyspark.sql import SparkSession
from IPython.display import Image as IPyImage, display

## 5. Configuration (derived from toggles)

**What this cell does:** assembles the `CONFIG` dict that downstream cells
read from. Table path and panel filename auto-rename based on
`BASE_FILE_FORMAT` so re-runs across formats produce side-by-side artifacts
on disk.

`BLOB_REFERENCE_CAST` is the SQL type the BLOB struct's `reference` field
needs to be cast to when it's null. The shape matches `HoodieSchema.Blob` in
`hudi-common`:
`struct<external_path:string, offset:bigint, length:bigint, managed:boolean>`.
We inline-cast `null` to this type in the INSERT (cell 12) so Spark's
analyzer doesn't trip on `null` having no inferable type.

In [5]:
CONFIG = {
    "dataset": "OxfordIIITPet",
    "table_path": f"/tmp/hudi_sql_{BASE_FILE_FORMAT}_pets",
    "table_name": f"pets_sql_{BASE_FILE_FORMAT}",
    "base_file_format": BASE_FILE_FORMAT,
    "n_samples": N_SAMPLES,
    "top_k": TOP_K,
    "embedding_model": EMBEDDING_MODEL,
    "output_dir": "./outputs",
    "panel_filename": f"hudi_sql_{BASE_FILE_FORMAT}_results.png",
}

BLOB_REFERENCE_CAST = "struct<external_path:string,offset:bigint,length:bigint,managed:boolean>"

for k, v in CONFIG.items():
    print(f"  {k:20s}: {v}")

  dataset             : OxfordIIITPet
  table_path          : /tmp/hudi_sql_parquet_pets
  table_name          : pets_sql_parquet
  base_file_format    : parquet
  n_samples           : 250
  top_k               : 5
  embedding_model     : mobilenetv3_small_100
  output_dir          : ./outputs
  panel_filename      : hudi_sql_parquet_results.png


## 6. Resolve Hudi (and optionally Lance) jars

**What this cell does:** resolves the absolute paths to the Hudi Spark
bundle and (if `BASE_FILE_FORMAT == "lance"`) the Lance Spark bundle.

**Defaults:**
- `HUDI_BUNDLE_JAR` → `~/Downloads/hudi-spark3.5-bundle_2.12-1.2.0-rc1.jar`
  (Apache 1.2.0-rc1 staging jar)
- `LANCE_BUNDLE_JAR` → `~/Downloads/lance-spark-bundle-3.5_2.12-0.4.0.jar`
  (Maven Central)

If you placed both jars in `~/Downloads/` per the parent
[`README.md`](../README.md) §1–2, no exports are needed. Otherwise set
`HUDI_BUNDLE_JAR` and/or `LANCE_BUNDLE_JAR` *before launching jupyter*
(SparkSessions can't pick up env-var changes mid-process).

In [6]:
import sys
from pathlib import Path

def default_hudi_bundle_jar() -> str:
    # Defaults to the Apache 1.2.0-rc1 staging jar in ~/Downloads/. Grab it with:
    #   curl -L -o ~/Downloads/hudi-spark3.5-bundle_2.12-1.2.0-rc1.jar \
    #     https://repository.apache.org/content/repositories/orgapachehudi-1176/org/apache/hudi/hudi-spark3.5-bundle_2.12/1.2.0-rc1/hudi-spark3.5-bundle_2.12-1.2.0-rc1.jar
    # Override via HUDI_BUNDLE_JAR env var to point at a locally built bundle.
    return str(Path.home() / "Downloads" / "hudi-spark3.5-bundle_2.12-1.2.0-rc1.jar")

def default_lance_bundle_jar() -> str:
    # Defaults to the Maven Central Lance 0.4.0 jar in ~/Downloads/. Grab it with:
    #   curl -L -o ~/Downloads/lance-spark-bundle-3.5_2.12-0.4.0.jar \
    #     https://repo1.maven.org/maven2/com/lancedb/lance-spark-bundle-3.5_2.12/0.4.0/lance-spark-bundle-3.5_2.12-0.4.0.jar
    return str(Path.home() / "Downloads" / "lance-spark-bundle-3.5_2.12-0.4.0.jar")

def resolve_jars(base_file_format: str) -> str:
    hudi_jar = os.getenv("HUDI_BUNDLE_JAR", default_hudi_bundle_jar())
    if not Path(hudi_jar).is_file():
        sys.exit(
            f"ERROR: HUDI_BUNDLE_JAR does not exist at {hudi_jar}\n"
            "Download the Apache 1.2.0-rc1 staging jar to ~/Downloads/ "
            "or set HUDI_BUNDLE_JAR=/abs/path/to/locally-built.jar "
            "before launching jupyter."
        )
    if base_file_format != "lance":
        return hudi_jar
    lance_jar = os.getenv("LANCE_BUNDLE_JAR", default_lance_bundle_jar())
    if not Path(lance_jar).is_file():
        sys.exit(
            f"ERROR: LANCE_BUNDLE_JAR does not exist at {lance_jar}\n"
            "Download lance-spark-bundle-3.5_2.12-0.4.0.jar from Maven Central "
            "to ~/Downloads/ or set LANCE_BUNDLE_JAR=/abs/path/to/jar "
            "before launching jupyter."
        )
    return f"{hudi_jar},{lance_jar}"

## 7. Spark session

**What this cell does:** boots a local-mode SparkSession with the four
Hudi-mandatory configs:

- `spark.jars` → the bundle path(s) from cell 6
- `spark.serializer` → `KryoSerializer` (Hudi internals require it)
- `spark.sql.extensions` → `HoodieSparkSessionExtension` (registers
  `hudi_vector_search`, `read_blob`, the BLOB struct constructor, and the
  `VECTOR(N)`/`BLOB` type parsers used in cell 11's DDL)
- `spark.sql.catalog.spark_catalog` → `HoodieCatalog` (so plain
  `CREATE TABLE … USING hudi` works without a separate catalog setup)

Plus `hoodie.read.blob.inline.mode = CONTENT` so reads materialize INLINE
blob bytes into the `data` field by default. (You'll see Spark warn
"Ignoring non-Spark config property" for this one — that's expected; it's
a Hudi-side option, not a Spark-side one. It still takes effect.)

In [7]:
jars = resolve_jars(CONFIG["base_file_format"])

spark = (
    SparkSession.builder
    .appName("Hudi-SQL-Vector-Blob-Demo")
    .config("spark.jars", jars)
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
    .config("spark.sql.extensions", "org.apache.spark.sql.hudi.HoodieSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.hudi.catalog.HoodieCatalog")
    .config("spark.sql.session.timeZone", "UTC")
    .config("hoodie.read.blob.inline.mode", "CONTENT")
    .config("spark.default.parallelism", "2")
    .config("spark.sql.shuffle.partitions", "2")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("✓ Spark session ready")

26/04/28 23:50:56 WARN Utils: Your hostname, mac.lan resolves to a loopback address: 127.0.0.1; using 192.168.86.21 instead (on interface en0)
26/04/28 23:50:56 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
26/04/28 23:50:57 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/28 23:50:57 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


✓ Spark session ready


## 8. Load Oxford-IIIT Pet (pure Python)

**What this cell does:** downloads the Oxford-IIIT Pet dataset via
`torchvision` (cached at `~/.cache/torchvision`), samples `N_SAMPLES`
images at random, encodes each one to PNG bytes, and emits a list of
dicts with bytes + metadata (image_id, breed, label, dimensions).

This is the source data both the embedding step (cell 9) and the BLOB
column (cell 12's INSERT) will draw from. Pure Python — no Spark involved
yet.

In [8]:
print(f"Loading dataset: Oxford-IIIT Pet ({CONFIG['n_samples']} samples)...")
root = os.path.expanduser("~/.cache/torchvision")
ds = OxfordIIITPet(root=root, split="trainval", download=True)
class_names = ds.classes

rng = np.random.default_rng()
n = min(CONFIG["n_samples"], len(ds))
indices = rng.choice(len(ds), size=n, replace=False)

data = []
for idx in indices:
    img, label = ds[int(idx)]
    img = img.convert("RGB")
    bio = io.BytesIO()
    img.save(bio, format="PNG")
    img_bytes = bio.getvalue()
    w, h = img.size
    category = class_names[label] if isinstance(class_names, list) else str(label)
    data.append({
        "image_id": f"pets_{int(idx):06d}",
        "category": category,
        "category_sanitized": category.replace("/", "_"),
        "label": int(label),
        "description": f"{category} from Oxford-IIIT Pet",
        "image_bytes_raw": img_bytes,
        "width": int(w),
        "height": int(h),
    })

print(f"✓ Loaded {len(data)} images")

Loading dataset: Oxford-IIIT Pet (250 samples)...
✓ Loaded 250 images


## 9. Embedding model + generate embeddings

**What this cell does:** loads a pretrained image classifier from `timm`
with `num_classes=0` (strips the classifier head, so `model(x)` returns
feature vectors instead of class logits), runs every loaded image through
it in a single batched forward pass, and L2-normalizes the resulting
vectors.

L2 normalization is what makes `cosine` distance reduce to
`1 - dot_product`, which is exactly what `hudi_vector_search`'s
`'cosine'` metric computes internally. Skipping it would still work but
the distances wouldn't match a textbook cosine.

`embedding_dim` (typically 1024 for `mobilenetv3_small_100`) feeds the
`VECTOR(N)` declaration in the next CREATE TABLE.

In [9]:
print(f"Loading embedding model: {CONFIG['embedding_model']}...")
model = timm.create_model(CONFIG["embedding_model"], pretrained=True, num_classes=0)
model.eval()
data_config = timm.data.resolve_model_data_config(model)
transform = timm.data.create_transform(**data_config, is_training=False)
print("✓ Model loaded")

print(f"Generating embeddings for {len(data)} images...")
images = [transform(Image.open(io.BytesIO(d["image_bytes_raw"])).convert("RGB"))
          for d in data]
batch = torch.stack(images)
with torch.no_grad():
    feats = model(batch).detach().cpu().numpy()
feats = normalize(feats)
for i, d in enumerate(data):
    d["embedding"] = feats[i].tolist()
embedding_dim = int(feats.shape[1])
print(f"✓ Generated embeddings (dimension: {embedding_dim})")

Loading embedding model: mobilenetv3_small_100...
✓ Model loaded
Generating embeddings for 250 images...
✓ Generated embeddings (dimension: 1024)


## 10. Stage Python data → Parquet → Spark temp view

**What this cell does:** writes the dataset (PNG bytes + embeddings + IDs +
labels) to a staging Parquet file on disk via PyArrow, then registers it
as a Spark temp view named `staging_pets`.

**Why not `spark.createDataFrame(...)` directly:** on local-mode Spark,
sustained binary streaming through PySpark's `PythonRDD` driver-to-executor
channel can saturate socket buffers on payloads of this size. PyArrow
writes Parquet in-process; Spark then reads the file back via its native
JVM Parquet reader. Same end state, no socket pressure.

The `INSERT INTO … SELECT … FROM staging_pets` in cell 12 will pull from
this view to populate the Hudi table.

In [10]:
STAGING_VIEW = "staging_pets"

arrow_schema = pa.schema([
    pa.field("image_id", pa.string(), nullable=False),
    pa.field("category", pa.string(), nullable=False),
    pa.field("category_sanitized", pa.string(), nullable=False),
    pa.field("label", pa.int32(), nullable=False),
    pa.field("description", pa.string(), nullable=True),
    pa.field("image_bytes_raw", pa.binary(), nullable=False),
    pa.field("width", pa.int32(), nullable=False),
    pa.field("height", pa.int32(), nullable=False),
    pa.field(
        "embedding",
        pa.list_(pa.field("element", pa.float32(), nullable=False),
                 list_size=embedding_dim),
        nullable=False,
    ),
])
columns = {
    "image_id": [d["image_id"] for d in data],
    "category": [d["category"] for d in data],
    "category_sanitized": [d["category_sanitized"] for d in data],
    "label": [int(d["label"]) for d in data],
    "description": [d.get("description") for d in data],
    "image_bytes_raw": [d["image_bytes_raw"] for d in data],
    "width": [int(d["width"]) for d in data],
    "height": [int(d["height"]) for d in data],
    "embedding": [d["embedding"] for d in data],
}
staging_path = f"/tmp/staging_{CONFIG['table_name']}_parquet.parquet"
pq.write_table(pa.table(columns, schema=arrow_schema), staging_path)
spark.read.parquet(staging_path).createOrReplaceTempView(STAGING_VIEW)
print(f"✓ Registered Spark temp view: {STAGING_VIEW}")

✓ Registered Spark temp view: staging_pets


## 11. CREATE TABLE — `VECTOR(N)` and `BLOB` are first-class

**What this cell does:** drops any pre-existing table by the same name,
then creates a fresh Hudi table with `embedding VECTOR(<dim>)` and
`image_bytes BLOB` declared as native column types.

`VECTOR(N)` and `BLOB` are Hudi 1.2.0 logical types — both the writer
and the reader understand them as first-class types instead of as nested
arrays or structs. The `VECTOR(N)` declaration also feeds the indexer
that `hudi_vector_search` uses at query time (cell 14).

**TBLPROPERTIES worth flagging:**
- `'hoodie.table.base.file.format'` — flips between `parquet` and `lance`
  per the toggle in cell 1.
- `'hoodie.write.record.merge.custom.implementation.classes' = 'org.apache.hudi.DefaultSparkRecordMerger'` —
  mandatory for Lance (flips the writer factory from AVRO to SPARK).
  Harmless for Parquet, so we leave it on always.

In [11]:
spark.sql(f"DROP TABLE IF EXISTS {CONFIG['table_name']}")

ddl = f"""
    CREATE TABLE {CONFIG['table_name']} (
        image_id            STRING,
        category            STRING,
        category_sanitized  STRING,
        label               INT,
        description         STRING,
        image_bytes         BLOB           COMMENT 'Pet image bytes (INLINE)',
        width               INT,
        height              INT,
        embedding           VECTOR({embedding_dim})
    ) USING hudi
    PARTITIONED BY (category_sanitized)
    LOCATION '{CONFIG['table_path']}'
    TBLPROPERTIES (
        primaryKey = 'image_id',
        preCombineField = 'image_id',
        type = 'cow',
        'hoodie.table.base.file.format' = '{CONFIG['base_file_format']}',
        'hoodie.write.record.merge.custom.implementation.classes' = 'org.apache.hudi.DefaultSparkRecordMerger'
    )
"""
print(ddl.strip())
spark.sql(ddl)
print(f"\n✓ Created table {CONFIG['table_name']} at {CONFIG['table_path']}")

CREATE TABLE pets_sql_parquet (
        image_id            STRING,
        category            STRING,
        category_sanitized  STRING,
        label               INT,
        description         STRING,
        image_bytes         BLOB           COMMENT 'Pet image bytes (INLINE)',
        width               INT,
        height              INT,
        embedding           VECTOR(1024)
    ) USING hudi
    PARTITIONED BY (category_sanitized)
    LOCATION '/tmp/hudi_sql_parquet_pets'
    TBLPROPERTIES (
        primaryKey = 'image_id',
        preCombineField = 'image_id',
        type = 'cow',
        'hoodie.table.base.file.format' = 'parquet',
        'hoodie.write.record.merge.custom.implementation.classes' = 'org.apache.hudi.DefaultSparkRecordMerger'
    )

✓ Created table pets_sql_parquet at /tmp/hudi_sql_parquet_pets


## 12. INSERT INTO ... SELECT — `named_struct` builds the BLOB INLINE struct

**What this cell does:** copies all rows from the staging temp view into
the Hudi table, wrapping the raw `image_bytes_raw` (bytes) into the
**INLINE BLOB struct** that Hudi expects on the wire.

The struct shape is fixed by `HoodieSchema.Blob`:

```
struct<
  type:      string,    -- "INLINE" or "OUT_OF_LINE"
  data:      binary,    -- the actual bytes (INLINE only)
  reference: struct<    -- the external pointer (OUT_OF_LINE only)
    external_path: string,
    offset:        bigint,
    length:        bigint,
    managed:       boolean
  >
>
```

For INLINE rows: `type='INLINE'`, `data=<bytes>`, `reference=null` (cast to
the right struct shape via `BLOB_REFERENCE_CAST` so Spark's analyzer accepts
the typeless null).

After the INSERT we sanity-count with `COUNT(image_id)`, *not* `COUNT(*)` —
naming a column ensures a non-empty projection that Hudi's Lance reader
handles correctly.

In [ ]:
insert = f"""
    INSERT INTO {CONFIG['table_name']}
    SELECT
        image_id,
        category,
        category_sanitized,
        label,
        description,
        named_struct(
            'type',      'INLINE',
            'data',      image_bytes_raw,
            'reference', cast(null as {BLOB_REFERENCE_CAST})
        ) AS image_bytes,
        width,
        height,
        embedding
    FROM {STAGING_VIEW}
"""
spark.sql(insert)

# COUNT(image_id), not COUNT(*): naming a column ensures a non-empty
# projection that Hudi's Lance reader handles correctly.
count = spark.sql(
    f"SELECT COUNT(image_id) AS c FROM {CONFIG['table_name']}"
).collect()[0]["c"]
print(f"✓ Inserted {count} rows")

## 13. Sample rows

**What this cell does:** quick sanity check that the INSERT actually
committed — selects a few `(image_id, category, description)` triples
from the table.

If this returns zero rows, something's wrong with the write — re-run
cell 12 and watch for write errors. If it returns rows but the breeds
look stuck on a single category, re-run cell 8 (the random sampling) to
get a more diverse slice.

In [ ]:
spark.sql(f"""
    SELECT image_id, category, description
    FROM {CONFIG['table_name']}
    LIMIT 5
""").show(truncate=False)

## 14. Vector search + `read_blob()` — both Hudi features in one query

**This is the headline cell.** It runs a single SQL query that composes
the two Hudi 1.2.0 features this notebook is about:

1. **`hudi_vector_search`** — a SQL Table-Valued Function. Given a table
   path, the VECTOR column name, a query vector (as `ARRAY(...)`), `k`,
   and a metric (`'cosine'`, `'l2'`, etc.), it returns the top-`k` rows
   ordered by `_hudi_distance`.
2. **`read_blob()`** — Hudi's polymorphic accessor that materializes a
   BLOB column to raw `binary`. With INLINE blobs (this notebook), it
   reads from the row's `data` field. With OUT_OF_LINE blobs (see
   [`01_blob_reader.ipynb`](01_blob_reader.ipynb)), it would resolve the
   `reference` to an external container file. **Same SQL either way —
   that's the point.**

Composing them in one SELECT:

```sql
SELECT image_id,
       category,
       read_blob(image_bytes) AS resolved_bytes,
       _hudi_distance
FROM hudi_vector_search('<table_path>', 'embedding',
                        ARRAY(<query_vector>), k+1, 'cosine')
ORDER BY _hudi_distance
```

We ask for `top_k + 1` because the query image itself is in the corpus
(`_hudi_distance ≈ 0`) and gets filtered out below. The query vector is
inlined into the SQL as a literal `ARRAY(...)` of floats — `hudi_vector_search`
requires the query vector as a constant expression, so a Python parameter
binding wouldn't work.

**What you'll see when this runs:** the chosen query image's `(image_id,
category)`, then a `Top K matches` list with similarity scores
(`1 - distance`). Higher similarity = closer match.

In [ ]:
# Pick a random image from the corpus as the query.
query_idx = np.random.randint(len(data))
query_item = data[query_idx]
print(f"Query: {query_item['image_id']} ({query_item['category']})")

# Inline the query vector as a SQL ARRAY literal.
array_literal = "ARRAY(" + ", ".join(f"{float(v)}" for v in query_item["embedding"]) + ")"

# Single composed query: hudi_vector_search + read_blob.
# The TVF returns the top-k rows; read_blob materializes the BLOB column to
# raw binary in the same SELECT — no second pass through the table needed.
query_sql = f"""
    SELECT image_id,
           category,
           read_blob(image_bytes) AS resolved_bytes,
           _hudi_distance
    FROM hudi_vector_search(
        '{CONFIG['table_path']}',
        'embedding',
        {array_literal},
        {CONFIG['top_k'] + 1},
        'cosine'
    )
    ORDER BY _hudi_distance
"""
rows = spark.sql(query_sql).collect()

# Filter out the query image itself (it's in the corpus → distance ≈ 0).
results = []
for row in rows:
    distance = float(row["_hudi_distance"])
    if distance < 0.001:
        continue
    if len(results) >= CONFIG["top_k"]:
        break
    resolved = row["resolved_bytes"]
    results.append({
        "image_id":   row["image_id"],
        "category":   row["category"],
        "image_bytes": bytes(resolved) if resolved is not None else b"",
        "similarity": 1.0 - distance,
    })

print(f"\n✓ Top {len(results)} matches:")
for i, r in enumerate(results, 1):
    print(f"  {i}. {r['image_id']:18s} {r['category']:30s} sim={r['similarity']:.3f}")

## 15. Visualize — query + top-K panel

**What this cell does:** builds a matplotlib figure with the query image
on the left and the top-K nearest neighbors to its right, each annotated
with its breed and similarity score (`1 - distance`).

The bytes for every panel image came back through `read_blob()` in cell 14
— neither the query image's bytes nor the result rows' bytes were ever
streamed through PySpark's `PythonRDD`. The panel is saved to
`outputs/<panel_filename>.png` and rendered inline below via
`IPython.display.Image`.

**For an ideal run:** the top-K should be visually similar to the query.
e.g. a Pug query returns other Pugs and Japanese Chins, with similarity
scores in the 0.4–0.6 range at N=250 (tighter at N=1000+). If similarity
is low across the board, the embedding model isn't finding signal —
either the dataset slice is too small or the model needs a domain-specific
fine-tune.

In [ ]:
out_dir = Path(CONFIG["output_dir"])
out_dir.mkdir(parents=True, exist_ok=True)

n_results = len(results)
fig, axes = plt.subplots(1, n_results + 1, figsize=(3 * (n_results + 1), 3.2))
query_img = Image.open(io.BytesIO(query_item["image_bytes_raw"])).convert("RGB")
axes[0].imshow(query_img)
axes[0].set_title(f"QUERY\n{query_item['category']}", fontweight="bold")
axes[0].axis("off")
for i, r in enumerate(results):
    img = Image.open(io.BytesIO(r["image_bytes"])).convert("RGB")
    axes[i + 1].imshow(img)
    axes[i + 1].set_title(f"{r['category']}\nSim: {r['similarity']:.3f}")
    axes[i + 1].axis("off")
plt.tight_layout()
panel_path = out_dir / CONFIG["panel_filename"]
plt.savefig(str(panel_path), dpi=150, bbox_inches="tight")
plt.close(fig)

print(f"✓ Panel saved: {panel_path}")
display(IPyImage(filename=str(panel_path)))

## 16. Stop Spark (optional)

**What this cell does:** shuts down the SparkSession and releases the
driver JVM.

**Skip this cell if you plan to flip toggles and Run All again** — the
existing SparkSession is reusable. Run it when you're done with the
notebook to free memory.

In [ ]:
spark.stop()
print("✓ Spark stopped")